In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("all_matches.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


In [3]:
pd.unique(df[['home_team','away_team']].values.ravel())

array(['Scotland', 'England', 'Wales', 'Ireland', 'Uruguay', 'Argentina',
       'Austria', 'Hungary', 'Bohemia', 'Belgium', 'France',
       'Switzerland', 'Netherlands', 'British Guiana',
       'Trinidad and Tobago', 'South Africa', 'Germany', 'Sweden',
       'Norway', 'Denmark', 'Italy', 'Chile', 'Finland', 'Luxembourg',
       'Russia', 'Philippines', 'China', 'Brazil', 'Suriname',
       'United States', 'Japan', 'Paraguay', 'Egypt', 'Greece', 'Spain',
       'Czechoslovakia', 'Yugoslavia', 'Estonia', 'Northern Ireland',
       'Costa Rica', 'El Salvador', 'Guatemala', 'Honduras', 'Poland',
       'Portugal', 'Romania', 'New Zealand', 'Australia', 'Latvia',
       'Mexico', 'Lithuania', 'Turkey', 'Aruba', 'Curaçao', 'Bulgaria',
       'Canada', 'Soviet Union', 'Haiti', 'Jamaica', 'Kenya', 'Uganda',
       'Bolivia', 'Azerbaijan', 'Armenia', 'Georgia', 'Peru',
       'British Honduras', 'Dutch East Indies', 'Barbados', 'Nicaragua',
       'Cuba', 'Faroe Islands', 'Iceland', 'Mart

In [4]:
from Confederations import get_confederation

all_teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
unknowns = [t for t in all_teams if get_confederation(t) == "Unknown"]
print(sorted(unknowns))

[]


In [5]:
def get_tournament_weight(tournament, home_team, away_team):
    t = tournament.strip()
    home_conf = get_confederation(home_team)
    away_conf = get_confederation(away_team)
    conf_priority = ["UEFA", "CONMEBOL", "CONCACAF", "CAF", "AFC", "OFC", "Unknown"]
    home_rank = conf_priority.index(home_conf)
    away_rank = conf_priority.index(away_conf)
    top_conf = home_conf if home_rank <= away_rank else away_conf

    # ── Tier 4.0 ──────────────────────────────────────────
    if t == "World Cup":
        return 4.0

    # ── Tier 3.0 ──────────────────────────────────────────
    if t in ["European Championship", "Copa America", "Copa América"]:
        return 3.0

    # ── Tier 2.5 ──────────────────────────────────────────
    if t in ["European Championship qual",
             "Copa America qualifier", "Copa América qualifier",
             "African Nations Cup", "Confederations Cup", "Olympic Games","Euro Ch q & Nordic Ch","Euro Ch q & British Ch","South American Champ","Mundialito"]:
        return 2.5

    if "European Nations League" in t and "CONCACAF" not in t:
        if "A" in t: return 2.5
        elif "B" in t: return 2.0
        elif "C" in t: return 1.75
        elif "D" in t: return 1.5
        else: return 2.5

    # ── Tier 2.0 ──────────────────────────────────────────
    if t == "World Cup qualifier":
        if top_conf in ["UEFA", "CONMEBOL"]: return 2.0
        elif top_conf == "CAF":              return 1.75
        elif top_conf == "CONCACAF":         return 1.75
        elif top_conf == "AFC":              return 1.5
        else:                                return 1.25  # OFC

    if t in ["African Nations Cup qualifier",
             "World Cup q & Nordic Ch", "World Cup q & British Ch",
             "NA Champ & WC qual", "World Cup and CONCACAF Ch q",
             "World Cup and Asian Cup qual", "World Cup and African Cup qual",
             "WC and Oce Cup q", "WC q and Oce Cup"]:
        return 2.0

    # ── Tier 1.75 ─────────────────────────────────────────
    if t in ["CONCACAF Championship", "Asian Cup",
             "CONCACAF Cup", "CONCACAF Series","Intercontinental Champ"]:
        return 1.75

    if "CONCACAF Nations League" in t:
        if "A" in t: return 1.75
        elif "B" in t: return 1.5
        elif "C" in t: return 1.25
        else: return 1.75

    # ── Tier 1.5 ──────────────────────────────────────────
    if t in ["Asian Cup qualifier", "CONCACAF Ch q",
             "Gulf Cup", "Arab Cup",
             "CECAFA Cup", "COSAFA Cup"]:
        return 1.5

    if t in ["Southeast Asian Champ", "South Asian Championship",
             "West Asian Championship", "East Asian Championship",
             "Central American Cup", "CFU Championship",
             "Copa Centenario", "Caribbean Cup",
             "Caribbean Championship","CONCACAF Ch q & Car Ch","CONCACAF Ch q & Car Ch PO","North American Champ","CONCACAF Ch & Car Ch q","Balkan & C European Champ"]:
        return 1.5

    # ── Tier 1.25 ─────────────────────────────────────────
    if t in ["Oceania Nations Cup", "Oceania Nations Cup qualifier",
             "Pacific Games", "Pacific Mini Games",
             "South Pacific Games", "South Pacific Mini Games",
             "Melanesian Cup", "Polynesian Cup"]:
        return 1.25

    # ── Keyword fallbacks ─────────────────────────────────
    t_lower = t.lower()

    if "qualifier" in t_lower or "qual" in t_lower:
        return 1.5

    if any(kw in t_lower for kw in ["championship", "cup", "tournament",
                                     "games", "friendly tournament"]):
        return 1.5

    # ── Tier 1.0 — Friendlies and everything else ─────────
    return 1.0
df['weight'] = df.apply(
    lambda row: get_tournament_weight(row['tournament'], row['home_team'], row['away_team']), 
    axis=1
)

In [6]:
df.groupby('tournament')['weight'].first().sort_values(ascending=False).tail(30)

tournament
CONCACAF Ch q & Car Ch PO        1.50
Friendship Games                 1.50
GANEFO Tournament                1.50
CONCACAF Ch q & Car Ch           1.50
CONCACAF Ch q & C Am Cup         1.50
Gossage Cup                      1.50
Dynasty Cup                      1.50
Pacific Mini Games               1.25
Pacific Games                    1.25
Oceania Nations Cup qualifier    1.25
Oceania Nations Cup              1.25
Melanesian Cup                   1.25
South Pacific Games              1.25
South Pacific Mini Games         1.25
Danube C & King Mihai C          1.00
Windward Islands Champ           1.00
Copa Cent Rev Mayo               1.00
Friendly                         1.00
Copa Circulo de la Prensa        1.00
FIFA Series                      1.00
Jakarta Anniversary Tourn        1.00
Copa Premio Honor Uruguayo       1.00
Artemio Franchi Trophy           1.00
Copa Ciudad de Mexico            1.00
Copa Paz del Chaco               1.00
Trans-Caucasian Champ            1.00
T

In [7]:
wc_qual = df[df['tournament'] == 'World Cup qualifier'][['home_team', 'away_team', 'weight']].sample(20, random_state=42)
print(wc_qual)

                     home_team            away_team  weight
35431                  Armenia                Spain    2.00
39080              South Korea           Uzbekistan    1.50
18790                    Japan            Hong Kong    1.50
20767                  Tunisia             Ethiopia    1.75
26548                 Paraguay              Uruguay    2.00
42948                  Czechia              Germany    2.00
43011                  Algeria               Zambia    1.75
23488                  Albania             Portugal    2.00
34393                  Ecuador              Bolivia    2.00
23551                   Angola             Zimbabwe    1.75
27889                    Ghana                Sudan    1.75
23697                    Kenya              Nigeria    1.75
31635                  England              Austria    2.00
40840  St Vincent & Grenadines               Guyana    1.75
42642              New Zealand                 Fiji    1.25
6850                Costa Rica  Trinidad

In [8]:
import importlib
import elo
importlib.reload(elo)
from elo import update_elo, get_elo, elo_ratings


df_sorted = df.sort_values('date').reset_index(drop=True)

for _, row in df_sorted.iterrows():
    update_elo(
        row['home_team'], row['away_team'],
        row['home_score'], row['away_score'],
        row['weight'], row['neutral']
    )
if "West Germany" in elo_ratings:
    elo_ratings["Germany"] = max(
        elo_ratings.get("Germany", 0),
        elo_ratings["West Germany"]
    )
    del elo_ratings["West Germany"]

if "Soviet Union" in elo_ratings:
    elo_ratings["Russia"] = max(
        elo_ratings.get("Russia", 0),
        elo_ratings["Soviet Union"]
    )
    del elo_ratings["Soviet Union"]

In [9]:
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
print(elo_df.sort_values('elo', ascending=False).head(20))

            team          elo
32         Spain  1651.191875
5      Argentina  1594.684017
10        France  1554.574859
16       Germany  1536.471299
1        England  1481.664114
44      Portugal  1472.160767
87       Ecuador  1457.922461
51        Turkey  1452.608560
166      Senegal  1452.216071
27        Brazil  1451.031690
84      Colombia  1443.322294
30         Japan  1437.919679
12   Netherlands  1431.156999
92       Croatia  1404.649163
18        Norway  1395.261868
49        Mexico  1388.173449
4        Uruguay  1386.957392
31      Paraguay  1371.302764
11   Switzerland  1362.855362
20         Italy  1356.566449
